# C2Align: MIMIC-III readmission prediction

This notebook contains the task-specific Specialist and Parallel branch prompts, their inference workflow, and Method 5 MoE_b (Dual KL Calib).


In [ ]:
import gc
import json
import os
import random
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

for _repository_root in (Path.cwd(), *Path.cwd().parents):
    if (_repository_root / "c2align_feature_policy.py").is_file():
        sys.path.insert(0, str(_repository_root))
        break
else:
    raise RuntimeError("Could not locate c2align_feature_policy.py.")
from c2align_feature_policy import numeric_feature_matrix

GLOBAL_SEED = 1
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)


## Local inference callback


In [ ]:
INFERENCE_CALLBACK = None


def configure_inference_callback(callback):
    global INFERENCE_CALLBACK
    INFERENCE_CALLBACK = callback


def call_llm(prompt, system_prompt=None, seed=GLOBAL_SEED, temperature=0.0):
    if INFERENCE_CALLBACK is None:
        raise RuntimeError(
            "Configure a local inference callback or load previously generated branch outputs."
        )
    raw = INFERENCE_CALLBACK(
        system_prompt=system_prompt,
        prompt=prompt,
        seed=seed,
        temperature=temperature,
    )
    if isinstance(raw, str):
        raw = raw.strip().removeprefix("```json").removesuffix("```").strip()
        raw = json.loads(raw)
    elif isinstance(raw, dict) and "response" in raw:
        raw = raw["response"]
    if not isinstance(raw, dict):
        raise TypeError("The inference callback must return a JSON object or a JSON string.")
    return {"response": raw, "success": True}


def _safe_call(prompt, system_prompt, seed=GLOBAL_SEED):
    try:
        return call_llm(prompt, system_prompt=system_prompt, seed=seed)
    except Exception:
        return {"response": {}, "success": False}


## Complete Specialist and Parallel prompts


In [ ]:
def get_specialist_prompts(target_task):
    if target_task != "readmission":
        raise ValueError("This notebook implements 30-day unplanned readmission.")

    task_objective = (
        "Predict 30-DAY UNPLANNED READMISSION: whether the patient will return "
        "within 30 days after discharge."
    )
    clinical_focus = (
        "Focus on chronic disease burden, frailty, unresolved discharge problems, "
        "discharge disposition, and barriers to adherence."
    )

    return {
        "spec_clinical": {
            "system": f"""You are a Clinical Status Specialist. Your goal is to review the notes to help {task_objective}
TASK: Extract acute symptoms, vital sign instability, and overall physical trajectory.
GUIDELINE: {clinical_focus}
OUTPUT JSON ONLY:
{{
    "key_symptoms": ["list of severe or unresolved symptoms"],
    "has_vital_instability": <bool>,
    "clinical_trajectory": "improving" | "stable" | "deteriorating"
}}"""
        },
        "spec_intervention": {
            "system": f"""You are a Treatment and Intervention Specialist. Your goal is to review the notes to help {task_objective}
TASK: Extract major life support, critical procedures, and high-risk medications.
OUTPUT JSON ONLY:
{{
    "key_interventions": ["list of major interventions"],
    "is_on_life_support": <bool>,
    "medication_risk_level": "high" | "medium" | "low"
}}"""
        },
        "spec_risk": {
            "system": f"""You are a Medical History Specialist. Your goal is to review the notes to help {task_objective}
TASK: Extract severe chronic conditions, historical risks, and critical laboratory or imaging findings.
OUTPUT JSON ONLY:
{{
    "chronic_conditions": ["list of severe chronic illnesses"],
    "critical_lab_imaging": ["list of severe abnormal findings"],
    "high_baseline_risk_flag": <bool>
}}"""
        },
        "prediction": {
            "system": f"""You are the Senior Diagnostician.
GOAL: {task_objective}
INPUT: Summaries from the Clinical, Intervention, and Risk Specialists.
OUTPUT JSON ONLY:
{{
    "prediction": <1 for YES or high risk, 0 for NO or low risk>,
    "probability": <float from 0.00 to 1.00 representing event risk>,
    "reasoning": "<brief reasoning citing specific evidence, at most 40 words>"
}}"""
        },
        "audit": {
            "system": f"""You are a Senior Chart Review Auditor.
GOAL: {task_objective}
INPUT: The original note, specialist summaries, and a preliminary prediction.
TASK: Audit the prediction for missed evidence, false assumptions, or unsupported statements.
OUTPUT JSON ONLY:
{{
    "prediction": <1 or 0>,
    "probability": <float from 0.00 to 1.00 representing finalized event risk>,
    "audit_reason": "<why the prediction was kept or changed, at most 40 words>"
}}"""
        },
    }


def get_parallel_prompt(target_task):
    if target_task != "readmission":
        raise ValueError("This notebook implements 30-day unplanned readmission.")
    return """You are a Senior Clinical Prediction Specialist. Analyze the complete patient record to predict 30-DAY UNPLANNED READMISSION after discharge. Focus on chronic disease burden, frailty, unresolved complications, discharge disposition, and barriers to adherence.
Return ONLY a valid JSON object with exactly these keys:
{
    "prediction": <1 if readmission will occur, otherwise 0>,
    "probability": <float from 0.00 to 1.00 representing readmission risk>,
    "reasoning": "<strongest supporting evidence, at most 40 words>"
}"""


## Specialist and Parallel branches


In [ ]:
def _clinical_context(row, target_task):
    context = {
        "age": row.get("admit_age"),
        "gender": row.get("gender"),
        "race": row.get("race"),
        "admission_type": row.get("admission_type"),
        "diagnoses": row.get("all_icd_codes"),
        "clinical_note": row.get("text"),
    }
    if target_task == "readmission":
        context["length_of_stay_days"] = row.get("los_days")
        context["discharge_location"] = row.get("discharge_location")
    return json.dumps(context, ensure_ascii=True)


def flatten_specialist_output(name, data):
    features = {}
    for key, value in data.items():
        feature_name = f"feat_{name}_{key}"
        if isinstance(value, bool):
            features[feature_name] = int(value)
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            features[feature_name] = float(value)
        elif isinstance(value, list):
            features[f"{feature_name}_count"] = len(value)
        elif isinstance(value, dict):
            for child_key, child_value in value.items():
                if isinstance(child_value, (bool, int, float)):
                    features[f"{feature_name}_{child_key}"] = float(child_value)
    return features


def run_specialist_branch(row, target_task, use_chart_review):
    prompts = get_specialist_prompts(target_task)
    patient_context = _clinical_context(row, target_task)
    specialist_keys = ["spec_clinical", "spec_intervention", "spec_risk"]
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [
            executor.submit(
                _safe_call,
                patient_context,
                prompts[key]["system"],
                GLOBAL_SEED + offset,
            )
            for offset, key in enumerate(specialist_keys)
        ]
    specialist_data = [
        result["response"] if result["success"] else {}
        for result in (future.result() for future in futures)
    ]
    summary = json.dumps(
        {
            "clinical_specialist": specialist_data[0],
            "intervention_specialist": specialist_data[1],
            "risk_specialist": specialist_data[2],
        },
        ensure_ascii=True,
    )
    prediction_result = _safe_call(
        f"Patient record: {patient_context}\nSpecialist findings: {summary}",
        prompts["prediction"]["system"],
    )
    prediction = (
        prediction_result["response"]
        if prediction_result["success"]
        else {"prediction": 0, "probability": 0.5, "reasoning": "No result."}
    )
    if use_chart_review:
        audit_result = _safe_call(
            (
                f"Patient record: {patient_context}\nSpecialist findings: {summary}\n"
                f"Preliminary prediction: {json.dumps(prediction, ensure_ascii=True)}"
            ),
            prompts["audit"]["system"],
        )
        if audit_result["success"]:
            prediction.update(audit_result["response"])
    output = {
        "mas_prediction": int(prediction.get("prediction", 0)),
        "mas_probability": float(prediction.get("probability", 0.5)),
        "reasoning": str(prediction.get("reasoning", "")),
        "audit_reason": str(prediction.get("audit_reason", "")),
        "specialist_findings": summary,
    }
    for name, data in zip(
        ["clinical", "intervention", "risk"],
        specialist_data,
    ):
        output.update(flatten_specialist_output(name, data))
    return output


def run_parallel_branch(row, target_task, num_agents):
    patient_context = _clinical_context(row, target_task)
    system_prompt = get_parallel_prompt(target_task)
    with ThreadPoolExecutor(max_workers=num_agents) as executor:
        futures = [
            executor.submit(
                _safe_call,
                patient_context,
                system_prompt,
                GLOBAL_SEED + offset * 123,
            )
            for offset in range(num_agents)
        ]
    responses = [
        result["response"]
        for result in (future.result() for future in futures)
        if result["success"]
    ]
    if not responses:
        return {"mas_prediction": 0, "mas_probability": 0.5, "reasoning": "No result."}
    probabilities = [float(item.get("probability", 0.5)) for item in responses]
    mean_probability = float(np.mean(probabilities))
    return {
        "mas_prediction": int(mean_probability >= 0.5),
        "mas_probability": mean_probability,
        "reasoning": " | ".join(str(item.get("reasoning", "")) for item in responses),
        "parallel_votes": json.dumps(
            [int(item.get("prediction", 0)) for item in responses]
        ),
    }


def generate_branch_outputs(
    patient_rows,
    target_task,
    branch,
    output_path,
    use_chart_review,
    num_parallel_agents,
):
    if branch not in {"specialist", "parallel"}:
        raise ValueError("branch must be 'specialist' or 'parallel'.")
    outputs = []
    for _, row in tqdm(patient_rows.iterrows(), total=len(patient_rows), desc=branch):
        if branch == "specialist":
            output = run_specialist_branch(row, target_task, use_chart_review)
        else:
            output = run_parallel_branch(row, target_task, num_parallel_agents)
        outputs.append(output)
    result = pd.concat(
        [patient_rows.reset_index(drop=True), pd.DataFrame(outputs)],
        axis=1,
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)
    return result


## Method 5 MoE_b (Dual KL Calib)


In [ ]:
def get_custom_xgb_obj_bin_dual_kl(
    specialist_probability,
    parallel_probability,
    alpha=0.05,
    beta_specialist=0.05,
    beta_parallel=0.05,
):
    specialist_probability = np.asarray(specialist_probability, dtype=float)
    parallel_probability = np.asarray(parallel_probability, dtype=float)

    def objective(y_true, raw_margin):
        probability = 1.0 / (1.0 + np.exp(-np.clip(raw_margin, -30.0, 30.0)))
        logistic_hessian = probability * (1.0 - probability)

        gradient_ce = probability - y_true
        hessian_ce = logistic_hessian

        gradient_brier = 2.0 * (probability - y_true) * logistic_hessian
        hessian_brier = 2.0 * logistic_hessian * (
            (probability - y_true) * (1.0 - 2.0 * probability) + logistic_hessian
        )

        gradient_specialist = probability - specialist_probability
        gradient_parallel = probability - parallel_probability

        gradient = (
            gradient_ce
            + alpha * gradient_brier
            + beta_specialist * gradient_specialist
            + beta_parallel * gradient_parallel
        )
        hessian = (
            hessian_ce
            + alpha * hessian_brier
            + beta_specialist * logistic_hessian
            + beta_parallel * logistic_hessian
        )
        return gradient, np.maximum(hessian, 1e-6)

    return objective


def probability_unit_interval(values):
    values = pd.to_numeric(values, errors="raise").to_numpy(dtype=float)
    if np.nanmax(values) > 1.0:
        values = values / 100.0
    if np.any((values < 0.0) | (values > 1.0)):
        raise ValueError("Branch probabilities must lie in [0, 1] or [0, 100].")
    return np.clip(values, 1e-6, 1.0 - 1e-6)


def align_branch_outputs(specialist, parallel, patient_col, label_col):
    required = {patient_col, label_col}
    for name, table in (("specialist", specialist), ("parallel", parallel)):
        missing = required.difference(table.columns)
        if missing:
            raise KeyError(f"{name} branch is missing columns: {sorted(missing)}")
        if table[patient_col].duplicated().any():
            raise ValueError(f"{name} branch contains duplicate patient identifiers.")

    parallel_copy = parallel.copy()
    parallel_copy = parallel_copy.rename(
        columns={
            column: f"parallel_{column}"
            for column in parallel_copy.columns
            if column not in {patient_col, label_col}
        }
    )
    aligned = specialist.merge(
        parallel_copy,
        on=[patient_col, label_col],
        how="inner",
        validate="one_to_one",
    )
    if len(aligned) != len(specialist) or len(aligned) != len(parallel):
        raise ValueError("Specialist and Parallel branch rows are not patient-aligned.")
    return aligned.sort_values(patient_col).reset_index(drop=True)

def make_method5_model(specialist_probability, parallel_probability):
    return xgb.XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_weight=1.0,
        random_state=GLOBAL_SEED,
        n_jobs=-1,
        eval_metric="logloss",
        objective=get_custom_xgb_obj_bin_dual_kl(
            specialist_probability,
            parallel_probability,
        ),
    )


def cross_validate_method5_moe_b(
    train_table,
    patient_col,
    label_col,
    specialist_probability_col,
    parallel_probability_col,
    model_feature_columns,
):
    features, feature_columns = numeric_feature_matrix(
        train_table,
        model_feature_columns,
        patient_col=patient_col,
        label_col=label_col,
    )
    labels = train_table[label_col].astype(int).to_numpy()
    groups = train_table[patient_col].to_numpy()
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)
    out_of_fold = np.zeros(len(train_table), dtype=float)

    for train_index, validation_index in folds.split(features, labels, groups):
        specialist_probability = probability_unit_interval(
            train_table.iloc[train_index][specialist_probability_col]
        )
        parallel_probability = probability_unit_interval(
            train_table.iloc[train_index][parallel_probability_col]
        )
        model = make_method5_model(specialist_probability, parallel_probability)
        model.fit(features.iloc[train_index], labels[train_index])
        out_of_fold[validation_index] = model.predict_proba(features.iloc[validation_index])[:, 1]

    return {
        "feature_columns": feature_columns,
        "feature_audit": features.attrs["feature_audit"],
        "probability": out_of_fold,
        "auc": roc_auc_score(labels, out_of_fold),
        "brier": brier_score_loss(labels, out_of_fold),
    }


def fit_and_evaluate_method5_moe_b(
    train_table,
    test_table,
    patient_col,
    label_col,
    specialist_probability_col,
    parallel_probability_col,
    model_feature_columns,
):
    overlap = set(train_table[patient_col]).intersection(set(test_table[patient_col]))
    if overlap:
        raise ValueError("Patient overlap detected between training and test sets.")

    train_features, feature_columns = numeric_feature_matrix(
        train_table,
        model_feature_columns,
        patient_col=patient_col,
        label_col=label_col,
    )
    test_features, _ = numeric_feature_matrix(
        test_table,
        feature_columns,
        patient_col=patient_col,
        label_col=label_col,
        imputation_values=train_features.median(numeric_only=True),
    )

    train_labels = train_table[label_col].astype(int).to_numpy()
    test_labels = test_table[label_col].astype(int).to_numpy()
    specialist_probability = probability_unit_interval(
        train_table[specialist_probability_col]
    )
    parallel_probability = probability_unit_interval(
        train_table[parallel_probability_col]
    )

    model = make_method5_model(specialist_probability, parallel_probability)
    model.fit(train_features, train_labels)
    probability = model.predict_proba(test_features)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    metrics = {
        "accuracy": accuracy_score(test_labels, prediction),
        "f1": f1_score(test_labels, prediction),
        "auc": roc_auc_score(test_labels, probability),
        "brier": brier_score_loss(test_labels, probability),
        "feature_columns": feature_columns,
        "feature_audit": train_features.attrs["feature_audit"],
    }
    return model, metrics, pd.DataFrame(
        {
            patient_col: test_table[patient_col].to_numpy(),
            "label": test_labels,
            "prediction": prediction,
            "probability": probability,
        }
    )


## Patient-aligned execution


In [ ]:
TARGET_TASK = "readmission"
USE_CHART_REVIEW = True
NUM_PARALLEL_AGENTS = 3
PATIENT_COLUMN = "subject_id"
LABEL_COLUMN = "label_numerical"
SPECIALIST_PROBABILITY_COLUMN = "mas_probability"
PARALLEL_PROBABILITY_COLUMN = "parallel_mas_probability"
MODEL_FEATURE_COLUMNS = (
    # Structured covariates available by the discharge prediction index.
    "admit_age",
    "los_days",
    # Numeric Specialist-branch features defined by the prompts above.
    "feat_clinical_key_symptoms_count",
    "feat_clinical_has_vital_instability",
    "feat_intervention_key_interventions_count",
    "feat_intervention_is_on_life_support",
    "feat_risk_chronic_conditions_count",
    "feat_risk_critical_lab_imaging_count",
    "feat_risk_high_baseline_risk_flag",
    # Continuous branch probabilities; hard predictions are excluded.
    SPECIALIST_PROBABILITY_COLUMN,
    PARALLEL_PROBABILITY_COLUMN,
)


def validate_readmission_split(train_table, test_table):
    if TARGET_TASK != "readmission":
        return
    if len(train_table) != 800 or len(test_table) != 200:
        raise ValueError("The documented readmission split contains 800 train and 200 test patients.")
    overlap = set(train_table[PATIENT_COLUMN]).intersection(set(test_table[PATIENT_COLUMN]))
    if overlap:
        raise ValueError("Patient overlap detected in the readmission split.")


def run_method5_from_csv(
    specialist_train_path,
    specialist_test_path,
    parallel_train_path,
    parallel_test_path,
):
    specialist_train = pd.read_csv(specialist_train_path)
    specialist_test = pd.read_csv(specialist_test_path)
    parallel_train = pd.read_csv(parallel_train_path)
    parallel_test = pd.read_csv(parallel_test_path)
    validate_readmission_split(specialist_train, specialist_test)

    train_table = align_branch_outputs(
        specialist_train,
        parallel_train,
        PATIENT_COLUMN,
        LABEL_COLUMN,
    )
    test_table = align_branch_outputs(
        specialist_test,
        parallel_test,
        PATIENT_COLUMN,
        LABEL_COLUMN,
    )
    cv_metrics = cross_validate_method5_moe_b(
        train_table,
        PATIENT_COLUMN,
        LABEL_COLUMN,
        SPECIALIST_PROBABILITY_COLUMN,
        PARALLEL_PROBABILITY_COLUMN,
        MODEL_FEATURE_COLUMNS,
    )
    model, test_metrics, predictions = fit_and_evaluate_method5_moe_b(
        train_table,
        test_table,
        PATIENT_COLUMN,
        LABEL_COLUMN,
        SPECIALIST_PROBABILITY_COLUMN,
        PARALLEL_PROBABILITY_COLUMN,
        MODEL_FEATURE_COLUMNS,
    )
    return model, cv_metrics, test_metrics, predictions


# Supply the four CSV paths and call run_method5_from_csv.
